In [10]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data import get_data_incremental
from src.signals import add_percentile_rank, add_zscore, classify_signal

In [ ]:
PERIOD = 180
PCT_FAVORABLE = 0.25
PCT_UNFAVORABLE = 0.75

df = get_data_incremental()
df = add_percentile_rank(df, period=PERIOD)
df = df.dropna(subset=["Percentile_Rank"]).copy()  # drop warm-up rows once, up front

df["Value_Signal"] = df["Percentile_Rank"].apply(
    lambda p: classify_signal(p, favorable=PCT_FAVORABLE, unfavorable=PCT_UNFAVORABLE)
)

is_favorable = df["Value_Signal"] == "Favorable"

# --- Option A: every Favorable day ---
favorable_days = df[is_favorable]
avg_rate_favorable_a = favorable_days["Close"].mean()

# --- Option B: one per episode (collapse consecutive Favorable streaks) ---
state_change = is_favorable != is_favorable.shift()
df["Episode_ID"] = state_change.cumsum()
favorable_episodes_first_day = df[is_favorable].groupby("Episode_ID", as_index=False).first()
avg_rate_favorable_b = favorable_episodes_first_day["Close"].mean()

# --- Option C: one per fixed 90-day period, fallback to period-end if never Favorable ---
def pick_transfer_day(period_df):
    favorable = period_df[period_df["Value_Signal"] == "Favorable"]
    row = favorable.iloc[0] if not favorable.empty else period_df.iloc[-1]
    row = row.copy()
    row["Transfer_Date"] = row.name
    return row

transfers_c = df.groupby(pd.Grouper(freq="90D")).apply(pick_transfer_day)
transfers_c = transfers_c.dropna(how="all").set_index("Transfer_Date")

avg_rate_favorable_c = transfers_c["Close"].mean()

# --- Baselines ---
rate_immediate = df["Close"].iloc[0]
random_rates = df["Close"].sample(n=5000, replace=True, random_state=42)
avg_rate_random = random_rates.mean()

# --- Comparison table ---
results = pd.DataFrame({
    "Strategy": [
        "A: every Favorable day",
        "B: one per episode",
        "C: one per 90-day period",
        "Immediate transfer",
        "Random day (avg of trials)",
    ],
    "Avg Rate Paid": [
        avg_rate_favorable_a,
        avg_rate_favorable_b,
        avg_rate_favorable_c,
        rate_immediate,
        avg_rate_random,
    ],
})
results["vs Random (%)"] = (results["Avg Rate Paid"] - avg_rate_random) / avg_rate_random * 100
print(results)

# --- Transaction cost sanity check ---
GBPUSD_SPREAD_PCT = 0.5   # rough estimate for the GBP/USD leg
USDIDR_SPREAD_PCT = 1.0   # rough estimate for the USD/IDR leg (EM currency, wider spread)
ESTIMATED_ROUNDTRIP_SPREAD_PCT = GBPUSD_SPREAD_PCT + USDIDR_SPREAD_PCT

print(f"\nEstimated round-trip spread for GBP/IDR: ~{ESTIMATED_ROUNDTRIP_SPREAD_PCT:.1f}%")
print("Does each strategy's edge survive this cost estimate?")
for _, row in results.iterrows():
    if row["Strategy"] == "Random day (avg of trials)":
        continue
    edge = -row["vs Random (%)"]  # flip sign: positive edge = strategy beat random
    verdict = "SURVIVES cost estimate" if edge > ESTIMATED_ROUNDTRIP_SPREAD_PCT else "does NOT clearly survive cost estimate"
    print(f"  {row['Strategy']:<30} edge={edge:+.2f}%  -> {verdict}")

[*********************100%***********************]  1 of 1 completed

Cache found, last date: 2026-09-11. Checking for new data...
Added 1 rows (updated cache total: 3045 rows).
                     Strategy  Avg Rate Paid  vs Random (%)
0      A: every Favorable day   17830.912588      -7.241577
1          B: one per episode   16197.449349     -15.739037
2    C: one per 90-day period   17502.384727      -8.950616
3          Immediate transfer   21927.000000      14.066733
4  Random day (avg of trials)   19222.957896       0.000000

Estimated round-trip spread for GBP/IDR: ~1.5%
Does each strategy's edge survive this cost estimate?
  A: every Favorable day         edge=+7.24%  -> SURVIVES cost estimate
  B: one per episode             edge=+15.74%  -> SURVIVES cost estimate
  C: one per 90-day period       edge=+8.95%  -> SURVIVES cost estimate
  Immediate transfer             edge=-14.07%  -> does NOT clearly survive cost estimate
